# Preprocess

In [2]:
import warnings
warnings.filterwarnings('ignore')

import scanpy as sc
import pandas as pd
import numpy as np
import muon

ImportError: cannot import name 'AlignedViewMixin' from 'anndata._core.aligned_mapping' (/home/qiaolibo/.local/lib/python3.9/site-packages/anndata/_core/aligned_mapping.py)

In [ ]:
import os 
os.chdir('/pool1/qiaolibo/R_Project/peak_gene/z_others/6_ann_visual/')

## RNA-seq preprocessing

In [4]:
rna = sc.read_h5ad('./processed/adata_rna.h5ad')
rna

AnnData object with n_obs × n_vars = 2462628 × 36601
    obs: 'dataset', 'barcode', 'sample', 'cellType'

In [5]:
rna.layers['counts'] = rna.X.copy()

In [6]:
sc.pp.normalize_total(rna, target_sum=1e4)
sc.pp.log1p(rna)

In [7]:
sc.pp.highly_variable_genes(rna, n_top_genes=4000, batch_key='sample')
rna

AnnData object with n_obs × n_vars = 2462628 × 36601
    obs: 'dataset', 'barcode', 'sample', 'cellType'
    var: 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'highly_variable_nbatches', 'highly_variable_intersection'
    uns: 'log1p', 'hvg'
    layers: 'counts'

In [8]:
rna_hvg = rna[:, rna.var.highly_variable].copy()
rna_hvg

AnnData object with n_obs × n_vars = 2462628 × 4000
    obs: 'dataset', 'barcode', 'sample', 'cellType'
    var: 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'highly_variable_nbatches', 'highly_variable_intersection'
    uns: 'log1p', 'hvg'
    layers: 'counts'

In [10]:
rna.write('../7_horizontal_RNA_ATAC/processed/rna.h5ad', compression='gzip')
rna_hvg.write('../7_horizontal_RNA_ATAC/processed/rna_hvg.h5ad', compression='gzip')

## ATAC-seq preprocessing

In [ ]:
atac = sc.read_h5ad('./processed/adata_atac.h5ad')
atac

### tfidf

In [4]:
atac.layers['counts'] = atac.X.copy()

In [5]:
muon.atac.pp.tfidf(atac, scale_factor=1e4)

In [6]:
sc.pp.highly_variable_genes(atac, n_top_genes=60000, batch_key='sample')
atac

AnnData object with n_obs × n_vars = 2462628 × 527459
    obs: 'dataset', 'barcode', 'sample', 'cellType'
    var: 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'highly_variable_nbatches', 'highly_variable_intersection'
    uns: 'hvg'
    layers: 'counts'

In [7]:
atac_hvf = atac[:, atac.var.highly_variable].copy()
atac_hvf

AnnData object with n_obs × n_vars = 2462628 × 60000
    obs: 'dataset', 'barcode', 'sample', 'cellType'
    var: 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'highly_variable_nbatches', 'highly_variable_intersection'
    uns: 'hvg'
    layers: 'counts'

In [8]:
atac.write('../7_horizontal_RNA_ATAC/processed/atac.h5ad', compression='gzip')
atac_hvf.write('../7_horizontal_RNA_ATAC/processed/atac_hvf.h5ad', compression='gzip')

### norm

In [ ]:
atac.layers['counts'] = atac.X.copy()

In [ ]:
sc.pp.normalize_total(atac, target_sum=1e4)
sc.pp.log1p(atac)

In [ ]:
sc.pp.highly_variable_genes(atac, n_top_genes=60000, batch_key='sample')
atac

In [ ]:
atac_hvg = atac[:, atac.var.highly_variable].copy()
atac_hvg

In [ ]:
atac.write('../7_horizontal_RNA_ATAC/processed/atac_norm.h5ad', compression='gzip')
atac_hvg.write('../7_horizontal_RNA_ATAC/processed/atac_norm_hvg.h5ad', compression='gzip')

## MultiVI

In [3]:
rna = sc.read_h5ad('../7_horizontal_RNA_ATAC/processed/rna_hvg.h5ad')
rna

AnnData object with n_obs × n_vars = 2462628 × 4000
    obs: 'dataset', 'barcode', 'sample', 'cellType'
    var: 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'highly_variable_nbatches', 'highly_variable_intersection'
    uns: 'hvg', 'log1p'
    layers: 'counts'

In [6]:
atac = sc.read_h5ad('../7_horizontal_RNA_ATAC/processed/atac_norm_hvg.h5ad')
atac

AnnData object with n_obs × n_vars = 2462628 × 60000
    obs: 'dataset', 'barcode', 'sample', 'cellType'
    var: 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'highly_variable_nbatches', 'highly_variable_intersection'
    uns: 'hvg', 'log1p'
    layers: 'counts'

In [7]:
import anndata as ad

In [8]:
merged_adata = ad.concat([rna, atac], axis=1) 
merged_adata

AnnData object with n_obs × n_vars = 2462628 × 64000
    var: 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'highly_variable_nbatches', 'highly_variable_intersection'
    layers: 'counts'

In [9]:
merged_adata.obs = rna.obs

In [10]:
merged_adata

AnnData object with n_obs × n_vars = 2462628 × 64000
    obs: 'dataset', 'barcode', 'sample', 'cellType'
    var: 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'highly_variable_nbatches', 'highly_variable_intersection'
    layers: 'counts'

In [11]:
merged_adata.var['modality'] = ['0'] * 4000 + ['1'] * 60000

In [12]:
merged_adata

AnnData object with n_obs × n_vars = 2462628 × 64000
    obs: 'dataset', 'barcode', 'sample', 'cellType'
    var: 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'highly_variable_nbatches', 'highly_variable_intersection', 'modality'
    layers: 'counts'

In [13]:
merged_adata.write('../7_horizontal_RNA_ATAC/processed/rna_atac_norm_merge.h5ad', compression='gzip')

## UINMF

In [14]:
data = sc.read_h5ad('../7_horizontal_RNA_ATAC/processed/rna_atac_tfidf_merge.h5ad')
data

AnnData object with n_obs × n_vars = 2462628 × 64000
    obs: 'dataset', 'barcode', 'sample', 'cellType'
    var: 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'highly_variable_nbatches', 'highly_variable_intersection', 'modality'
    layers: 'counts'

In [19]:
norm_data = sc.read_h5ad('../7_horizontal_RNA_ATAC/processed/rna_atac_norm_merge.h5ad')
norm_data

AnnData object with n_obs × n_vars = 2462628 × 64000
    obs: 'dataset', 'barcode', 'sample', 'cellType'
    var: 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'highly_variable_nbatches', 'highly_variable_intersection', 'modality'
    layers: 'counts'

In [33]:
data[:,4000:64000].X

<2462628x60000 sparse matrix of type '<class 'numpy.float64'>'
	with 140583274 stored elements in Compressed Sparse Row format>

In [34]:
norm_data[:,4000:64000].X

<2462628x60000 sparse matrix of type '<class 'numpy.float32'>'
	with 2079593942 stored elements in Compressed Sparse Row format>

In [36]:
2079593942/140583274

14.79261282533511

In [12]:
from scipy.io import mmwrite
from scipy.sparse import csr_matrix

# 假设稀疏矩阵 sparse_matrix 为 csr_matrix 格式
mmwrite("../7_horizontal_RNA_ATAC/processed/uinmf/matrix_count.mtx", data.layers['counts'])

In [18]:
cell = pd.DataFrame(data.obs_names)
cell

,0
0,Chen_Nat_Methods_2022-AAACAGCCAACTGGCT-lane1
1,Chen_Nat_Methods_2022-AAACATGCACAATTAC-lane1
2,Chen_Nat_Methods_2022-AAACATGCAGCACGTT-lane1
3,Chen_Nat_Methods_2022-AAACATGCATATTGAC-lane1
4,Chen_Nat_Methods_2022-AAACCAACAATAGCAA-lane1
...,...
2462623,Weng_Nature_2024_7-TTTGTGTTCGTAATCA-1
2462624,Weng_Nature_2024_7-TTTGTTGGTAGGATTT-1
2462625,Weng_Nature_2024_7-TTTGTTGGTCATTACC-1
2462626,Weng_Nature_2024_7-TTTGTTGGTGTTAAAC-1


In [19]:
cell.to_csv('../7_horizontal_RNA_ATAC/processed/uinmf/cell_norm.csv')

In [20]:
gene = pd.DataFrame(data.var_names)
gene

,0
0,C1orf159
1,TNFRSF18
2,TNFRSF4
3,PRDM16
4,AL365255.1
...,...
63995,chrX-155394783-155395283
63996,chrX-155611165-155611665
63997,chrX-155611723-155612223
63998,chrX-155632392-155632892


In [21]:
gene.to_csv('../7_horizontal_RNA_ATAC/processed/uinmf/feature_norm.csv')

# RunMultiVI

In [3]:
from scipy.io import mmread
from scipy.sparse import csr_matrix
import anndata as ad
import pandas as pd
import scvi
import numpy as np
import scanpy as sc
scvi.settings.seed = 420
import os, sys, time

Global seed set to 0
/home/qiaolibo/miniconda3/envs/mtg/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Global seed set to 420


In [2]:
dataset_id = "Dataset1/"
data_path = "../dataset/Horizontal/RNA_ATAC/"+dataset_id ## path to raw data

for i in range(1,3):
    batch_num = str(i)
    path = data_path+"batch"+batch_num
    cell_names = pd.read_csv(path+'/RNA/barcodes.tsv', sep = '\t', header=None, index_col=None)
    cell_names.columns =  ['cell_ids']
    X = csr_matrix(mmread(path+'/RNA/matrix.mtx').T)
    gene_names = pd.read_csv(path+'/RNA/genes.tsv', sep = '\t',  header=None, index_col=None)
    gene_names.columns =  ['gene_ids']
    locals()['adata_RNA'+str(i)] = ad.AnnData(X, obs=pd.DataFrame(index=cell_names.cell_ids), var=pd.DataFrame(index = gene_names.gene_ids))
    locals()['adata_RNA'+str(i)].var_names_make_unique()

    # peak information
    cell_names = pd.read_csv(path+'/ATAC/barcodes.tsv', sep = '\t', header=None, index_col=None)
    cell_names.columns =  ['cell_ids']
    X = csr_matrix(mmread(path+'/ATAC/matrix.mtx').T)
    peak_name = pd.read_csv(path+'/ATAC/peaks.tsv',header=None,index_col=None)
    peak_name.columns = ['peak_ids']
    locals()['adata_ATAC'+str(i)]  = ad.AnnData(X, obs=pd.DataFrame(index=cell_names.cell_ids), var=pd.DataFrame(index = peak_name.peak_ids))

del X

In [3]:
adata_RNA = ad.concat([adata_RNA1,adata_RNA2])
adata_RNA.obs['batch'] = ['batch1']*adata_RNA1.n_obs + ['batch2']*adata_RNA2.n_obs
sc.pp.highly_variable_genes(
    adata_RNA,
    batch_key="batch",
    flavor="seurat_v3",
    n_top_genes=4000,
    subset=True
)
adata_RNA1 = adata_RNA[adata_RNA.obs['batch'] == 'batch1'].copy()
adata_RNA2 = adata_RNA[adata_RNA.obs['batch'] == 'batch2'].copy()

for i in range(1,3):
    locals()['adata_paired'+str(i)] = ad.concat([eval('adata_RNA'+str(i)), eval('adata_ATAC'+str(i))], merge = "same",axis=1)
    eval('adata_paired'+str(i)).obs_keys
    modality = ['Gene Expression']*eval('adata_RNA'+str(i)).shape[1]+['Peaks']*eval('adata_ATAC'+str(i)).shape[1]
    locals()['adata_paired'+str(i)].var['modality']=modality

adata_mvi = scvi.data.organize_multiome_anndatas(adata_paired1, adata_paired2)
del adata_paired1, adata_paired2

adata_mvi = adata_mvi[:, adata_mvi.var["modality"].argsort()].copy()

/home/qiaolibo/miniconda3/envs/mtg/lib/python3.10/site-packages/scvi/data/_preprocessing.py:328: FutureWarning: Use anndata.concat instead of AnnData.concatenate, AnnData.concatenate is deprecated and will be removed in the future. See the tutorial for concat at: https://anndata.readthedocs.io/en/latest/concatenation.html
  return multi_anndata.concatenate(other, join="outer", batch_key=modality_key)


In [4]:
adata_mvi

AnnData object with n_obs × n_vars = 13383 × 70447
    obs: 'modality'
    var: 'modality'

In [6]:
adata_mvi.X

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 74139760 stored elements and shape (13383, 70447)>

## test

In [3]:
import torch

In [6]:
model = torch.load("../results/Dataset1/model_mtv.pth")

/tmp/ipykernel_1315857/4098047714.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model = torch.load("../results/Dataset1/model_mtv.pth")


In [7]:
model

MultiVI Model with INPUTS: n_genes:4000, n_regions:66447, n_proteins:0
n_hidden: 257, n_latent: 16, n_layers_encoder: 2, n_layers_decoder: 2 , dropout_rate: 0.1, latent_distribution: 
normal, deep injection: False, gene_likelihood: zinb, gene_dispersion:gene, Mod.Weights:equal, 
Mod.Penalty:Jeffreys, protein_dispersion:protein
Training status: Trained
Model's adata is minified?: False

In [16]:
adata_mvi.obsm['latent'] = model.get_latent_representation()

In [19]:
meta_1 = pd.read_csv('../dataset/Horizontal/RNA_ATAC/Dataset1/Dataset1_batch1_celltype.csv',
                    index_col=0)
meta_2 = pd.read_csv('../dataset/Horizontal/RNA_ATAC/Dataset1/Dataset1_batch2_celltype.csv',
                    index_col=0)
meta = pd.concat([meta_1, meta_2])
meta

,celltype
Batch1_AAACAGCCAAAGCGGC-1,CD4 Naive
Batch1_AAACAGCCAACAGGAT-1,CD4 TCM
Batch1_AAACAGCCAAGGTACG-1,CD4 Naive
Batch1_AAACAGCCAGGATGGC-1,gdT
Batch1_AAACAGCCATTCAGCA-1,CD4 TCM
...,...
Batch2_TTTGTGTTCGACAAAG-2,CD8 Naive
Batch2_TTTGTTGGTAGCCATA-2,CD8 Naive
Batch2_TTTGTTGGTAGCCTAA-2,Treg
Batch2_TTTGTTGGTGCAATAT-2,CD4 Naive


In [22]:
adata_mvi.obs['celltype'] = meta['celltype'].tolist()

In [23]:
sc.pp.neighbors(adata_mvi, use_rep='latent')
sc.tl.umap(adata_mvi)
latent = adata_mvi.obsm['latent'].copy()

In [26]:
adata_mvi

AnnData object with n_obs × n_vars = 13383 × 70447
    obs: 'modality', 'celltype'
    var: 'modality'
    uns: 'neighbors', 'umap'
    obsm: 'latent', 'X_umap'
    obsp: 'distances', 'connectivities'

In [ ]:
sc.pl.umap(adata_mvi, color=['modality', 'celltype'])

In [30]:
adata_mvi.write_h5ad('../results/Dataset1/adata_mtv.h5ad', compression='gzip')

In [29]:
np.savetxt("../results/Dataset1/latent_mtv.csv", latent, delimiter=',')

## test_blood

In [6]:
os.chdir('../')

In [7]:
adata_mvi = sc.read_h5ad('../results/blood/adata_mtv.h5ad')
adata_mvi

AnnData object with n_obs × n_vars = 2462628 × 64000
    obs: 'dataset', 'barcode', 'sample', 'cellType'
    var: 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'highly_variable_nbatches', 'highly_variable_intersection', 'modality'
    uns: 'neighbors', 'umap'
    obsm: 'X_umap', 'latent'
    layers: 'counts'
    obsp: 'connectivities', 'distances'

In [14]:
a = pd.DataFrame(adata_mvi.obs['cellType'].value_counts())

In [15]:
a

,count
cellType,
notAvailable,349124
CD4 Memory,226556
NK,170535
CD4,153153
CD8,134059
...,...
15_CLP.2,18
14_Unk,11
04_Early.Baso,9


In [7]:
data = pd.read_csv('/pool1/qiaolibo/R_Project/peak_gene/z_others/6_ann_visual/visual/sample_all.csv', index_col=0)
data['sample_'] = data.index.tolist()
cell = adata.obs
all = pd.merge(cell, data, left_on='sample', right_on='sample_')
all.index = adata.obs_names.tolist()

In [8]:
adata_mvi.obs = all

In [ ]:
sc.pl.umap(adata_mvi, color='technology')

In [ ]:
sc.pl.umap(adata_mvi, color='dataset')

In [ ]:
sc.pl.umap(adata_mvi, color='level_1')

In [ ]:
sc.pl.umap(adata_mvi, color='level_2')

In [ ]:
sc.pl.umap(adata_mvi, color='level_3')

## marker gene

In [ ]:
sc.pl.umap(adata_mvi, color=['CD79A', 'MS4A1'])

In [ ]:
sc.pl.umap(adata_mvi, color=['SPINK2', 'CRHBP'])

In [ ]:
sc.pl.umap(adata_mvi, color=['CD8A', 'CD3E'])

# uinmf_test

In [1]:
import scanpy as sc
import pandas as pd
import numpy as np

In [2]:
import os
os.chdir('/pool1/qiaolibo/R_Project/peak_gene/z_others/')

In [4]:
data = sc.read_h5ad('6_ann_visual/processed/adata_rna_celltypist_cellhint.h5ad')
data

AnnData object with n_obs × n_vars = 2462628 × 36601
    obs: 'dataset', 'barcode', 'sample', 'cellType', 'predicted_labels', 'over_clustering', 'majority_voting', 'conf_score', 'level_1', 'level_2', 'level_3'
    uns: 'level_1_colors', 'level_2_colors', 'log1p', 'neighbors', 'over_clustering', 'umap'
    obsm: 'X_pca', 'X_umap'
    obsp: 'connectivities', 'distances'

In [5]:
data.obsm['X_umap']

array([[ 1.0547097 ,  0.25992054],
       [-1.213523  ,  0.20383517],
       [ 0.4126014 , -0.98926085],
       ...,
       [ 0.82956123,  8.809223  ],
       [ 4.491045  , -4.0994086 ],
       [ 4.4805875 ,  3.195538  ]], dtype=float32)

## uinmf umap

In [68]:
umap = pd.read_csv('7_horizontal_RNA_ATAC/results/blood/uinmf_umap.csv', index_col=0)
umap

,UMAP_1,UMAP_2
10x_datasets_1-AAACAGCCAACAACAA-1,0.751791,1.456606
10x_datasets_1-AAACAGCCACCGGCTA-1,-1.250433,-5.877112
10x_datasets_1-AAACAGCCAGGACACA-1,-8.077833,-3.793774
10x_datasets_1-AAACAGCCATCCTAGA-1,0.272803,2.816770
10x_datasets_1-AAACATGCAAAGGTAC-1,-3.940582,7.429171
...,...,...
Wimmers_Cell_2023-TTTGTCTAGTTGTCCC-76_CC0829,-2.584904,-4.101810
Wimmers_Cell_2023-TTTGTGGCAGAGAGCC-76_CC0829,5.918900,-1.018441
Wimmers_Cell_2023-TTTGTGGCATTGCAGC-76_CC0829,7.691581,-3.857392
Wimmers_Cell_2023-TTTGTGGCATTGCGAC-76_CC0829,-5.153537,-6.902580


In [69]:
uma = umap.reindex(data.obs_names)
uma

,UMAP_1,UMAP_2
Chen_Nat_Methods_2022-AAACAGCCAACTGGCT-lane1,5.574091,-5.966500
Chen_Nat_Methods_2022-AAACATGCACAATTAC-lane1,10.767044,-5.040112
Chen_Nat_Methods_2022-AAACATGCAGCACGTT-lane1,9.191527,-3.379367
Chen_Nat_Methods_2022-AAACATGCATATTGAC-lane1,7.728632,-3.735427
Chen_Nat_Methods_2022-AAACCAACAATAGCAA-lane1,11.255870,-5.989583
...,...,...
Weng_Nature_2024_7-TTTGTGTTCGTAATCA-1,-3.215269,-5.614422
Weng_Nature_2024_7-TTTGTTGGTAGGATTT-1,-0.426363,-4.232681
Weng_Nature_2024_7-TTTGTTGGTCATTACC-1,4.087893,0.513445
Weng_Nature_2024_7-TTTGTTGGTGTTAAAC-1,-10.017323,-1.518041


In [75]:
data.obsm['X_umap'] = uma.to_numpy()

In [ ]:
sc.pl.umap(data, color='dataset')

In [ ]:
sc.pl.umap(data, color='level_1')

In [ ]:
sc.pl.umap(data, color='level_2')

In [ ]:
sc.pl.umap(data, color='level_3')

## umap_k30

In [6]:
umap = pd.read_csv('7_horizontal_RNA_ATAC/results/blood/uinmf_umap_k30.csv', index_col=0)
umap

,UMAP_1,UMAP_2
10x_datasets_1-AAACAGCCAACAACAA-1,2.626940,-0.087710
10x_datasets_1-AAACAGCCACCGGCTA-1,4.885952,1.895512
10x_datasets_1-AAACAGCCAGGACACA-1,-0.325860,-1.618190
10x_datasets_1-AAACAGCCATCCTAGA-1,-6.424448,-0.358324
10x_datasets_1-AAACATGCAAAGGTAC-1,-13.537602,-5.436432
...,...,...
Wimmers_Cell_2023-TTTGTCTAGTTGTCCC-76_CC0829,-12.805869,-7.109150
Wimmers_Cell_2023-TTTGTGGCAGAGAGCC-76_CC0829,1.761742,-1.334980
Wimmers_Cell_2023-TTTGTGGCATTGCAGC-76_CC0829,1.314012,11.311363
Wimmers_Cell_2023-TTTGTGGCATTGCGAC-76_CC0829,-1.135580,-4.397860


In [7]:
uma = umap.reindex(data.obs_names)
uma

,UMAP_1,UMAP_2
Chen_Nat_Methods_2022-AAACAGCCAACTGGCT-lane1,0.119360,2.804295
Chen_Nat_Methods_2022-AAACATGCACAATTAC-lane1,2.871486,-1.899806
Chen_Nat_Methods_2022-AAACATGCAGCACGTT-lane1,0.122624,-2.493524
Chen_Nat_Methods_2022-AAACATGCATATTGAC-lane1,0.229963,-4.894352
Chen_Nat_Methods_2022-AAACCAACAATAGCAA-lane1,1.023856,-5.494980
...,...,...
Weng_Nature_2024_7-TTTGTGTTCGTAATCA-1,-4.055349,5.624380
Weng_Nature_2024_7-TTTGTTGGTAGGATTT-1,-0.968586,4.806017
Weng_Nature_2024_7-TTTGTTGGTCATTACC-1,4.525038,-2.686464
Weng_Nature_2024_7-TTTGTTGGTGTTAAAC-1,-2.268269,-4.639720


In [8]:
data.obsm['X_umap'] = uma.to_numpy()

In [11]:
sample = pd.read_csv('/pool1/qiaolibo/R_Project/peak_gene/z_others/6_ann_visual/visual/sample_all.csv', index_col=0)
sample['sample_'] = sample.index.tolist()
cell = data.obs
all = pd.merge(cell, sample, left_on='sample', right_on='sample_')
all.index = data.obs_names.tolist()

In [12]:
data.obs = all

In [ ]:
sc.pl.umap(data, color='technology')

In [ ]:
sc.pl.umap(data, color='level_1')

In [ ]:
sc.pl.umap(data, color='level_2')

In [ ]:
sc.pl.umap(data, color='level_3')